In [1]:
import glob
import re

ctrl_vars = {'thl', 'qt', 'qr', 'qs', 'qg', 'u', 'v', 'w'}

class TP_perf:
    def __init__(self, name):
        self.name = name

        # Input: full 3D only.
        self.n_ctrl_in = 0
        self.tp_ctrl_in = 0

        # Output: full 3D restart files.
        self.n_ctrl_out = 0
        self.tp_ctrl_out = 0

        # Output: reduced 3D files.
        self.n_3d = 0
        self.tp_3d = 0

        # Output: xz cross-sections.
        self.n_xz = 0
        self.tp_xz = 0

        # Output: full xy cross-sections.
        self.n_xy = 0
        self.tp_xy = 0

        # Output: reduced xy cross-sections.
        self.n_xy_c = 0
        self.tp_xy_c = 0

    def add(self, name, tp_value):
        setattr(self, f'n_{name}', getattr(self, f'n_{name}') + 1)
        setattr(self, f'tp_{name}', getattr(self, f'tp_{name}') + tp_value)

    def mean(self, name):
        n = getattr(self, f'n_{name}')
        return getattr(self, f'tp_{name}') / n if n > 0 else None

    def print(self):
        for name in ('ctrl_in', 'ctrl_out', '3d', 'xz', 'xy', 'xy_c'):
            n = getattr(self, f'n_{name}')
            tp = getattr(self, f'tp_{name}')
            if n > 0:
                print(f'{name}: mean TP={tp/n:.2f} GB/s ({n} samples.)')
            else:
                print(f'{name}: no timings...')


files = glob.glob('out_io/*.stdout')
files.sort()

tp_instances = []

for stdout in files:
    tp = TP_perf(stdout)

    with open(stdout, 'r') as f:
        for l in f.readlines():
            if 'GB/s' not in l:
                continue

            tp_val = float(re.search(r'([\d.]+) GB/s', l).group(1))

            if l.startswith('Saving'):
                varname = re.search(r'Saving "(\w+)\.', l).group(1)
                if varname in ctrl_vars:
                    tp.add('ctrl_out', tp_val)
            elif l.startswith('Loading'):
                varname = re.search(r'Loading "(\w+)\.', l).group(1)
                if varname in ctrl_vars:
                    tp.add('ctrl_in', tp_val)
            elif '.xy_c.' in l:
                tp.add('xy_c', tp_val)
            elif '.xy.' in l:
                tp.add('xy', tp_val)
            elif '.xz.' in l:
                tp.add('xz', tp_val)
            else:
                tp.add('3d', tp_val)

    tp_instances.append(tp)

    print('-----------------')
    print(stdout)
    print('-----------------')
    tp.print()

-----------------
out_io/200_128_16x4_0_16x4.stdout
-----------------
ctrl_in: mean TP=8.76 GB/s (8 samples.)
ctrl_out: mean TP=8.31 GB/s (8 samples.)
3d: mean TP=4.23 GB/s (24 samples.)
xz: mean TP=6.03 GB/s (10 samples.)
xy: mean TP=0.48 GB/s (48 samples.)
xy_c: mean TP=0.08 GB/s (16 samples.)
-----------------
out_io/200_128_16x4_1_16x4.stdout
-----------------
ctrl_in: mean TP=6.16 GB/s (8 samples.)
ctrl_out: mean TP=6.69 GB/s (8 samples.)
3d: mean TP=4.52 GB/s (24 samples.)
xz: mean TP=4.18 GB/s (10 samples.)
xy: mean TP=0.46 GB/s (48 samples.)
xy_c: mean TP=0.09 GB/s (16 samples.)
-----------------
out_io/200_128_16x4_2_16x4.stdout
-----------------
ctrl_in: mean TP=8.07 GB/s (8 samples.)
ctrl_out: mean TP=9.40 GB/s (8 samples.)
3d: mean TP=4.04 GB/s (24 samples.)
xz: mean TP=6.81 GB/s (10 samples.)
xy: mean TP=0.46 GB/s (48 samples.)
xy_c: mean TP=0.07 GB/s (16 samples.)
-----------------
out_io/200_128_16x4_3_16x4.stdout
-----------------
ctrl_in: mean TP=8.14 GB/s (8 samples.)

In [3]:
import numpy as np

for res in (200, 400, 800):
    runs = [tp for tp in tp_instances if tp.name.startswith(f'out_io/{res}_')]
    print(f'--- Resolution {res} m ({len(runs)} runs) ---')
    for name in ('ctrl_in', 'ctrl_out', '3d', 'xz', 'xy', 'xy_c'):
        vals = [tp.mean(name) for tp in runs if tp.mean(name) is not None]
        if vals:
            print(f'  {name:10s}: mean={np.mean(vals):.2f} median={np.median(vals):.2f} min={np.min(vals):.2f}  max={np.max(vals):.2f} GB/s')
        else:
            print(f'  {name:10s}: no timings...')
    print()

--- Resolution 200 m (5 runs) ---
  ctrl_in   : mean=7.53 median=8.07 min=6.16  max=8.76 GB/s
  ctrl_out  : mean=7.81 median=7.61 min=6.69  max=9.40 GB/s
  3d        : mean=4.22 median=4.23 min=3.98  max=4.52 GB/s
  xz        : mean=5.64 median=6.03 min=4.18  max=6.81 GB/s
  xy        : mean=0.46 median=0.46 min=0.43  max=0.49 GB/s
  xy_c      : mean=0.08 median=0.08 min=0.07  max=0.10 GB/s

--- Resolution 400 m (5 runs) ---
  ctrl_in   : mean=5.58 median=5.21 min=4.37  max=7.09 GB/s
  ctrl_out  : mean=7.25 median=7.23 min=6.80  max=7.73 GB/s
  3d        : mean=3.58 median=3.51 min=3.37  max=4.06 GB/s
  xz        : mean=2.50 median=2.44 min=1.66  max=3.56 GB/s
  xy        : mean=0.42 median=0.43 min=0.39  max=0.44 GB/s
  xy_c      : mean=0.10 median=0.11 min=0.07  max=0.13 GB/s

--- Resolution 800 m (5 runs) ---
  ctrl_in   : mean=14.28 median=13.69 min=12.86  max=16.70 GB/s
  ctrl_out  : mean=3.96 median=3.93 min=3.74  max=4.22 GB/s
  3d        : mean=2.43 median=2.43 min=2.16  max=2.